This code is specifically to be run after DLT pipeline populates data in bronze layer tables for **chunk2.csv** files specifically for **city_time_series** and **zip_time_series** tables 

In [0]:
# 1. Configuration
try:
  from schema_config import (
    source_schema,
    target_schema,
    tables
  )
except:
  tables = ["city_time_series", "zip_time_series"]
  source_schema = "workspace.default"
  target_schema = "data_bronze.bronze"

for t_name in tables:
    source_table = f"{source_schema}.{t_name}"
    target_table = f"{target_schema}.{t_name}"
    
    print(f"Processing vertical append for {t_name}...")

    # 2. Get intersection of columns to avoid "Unresolved Expression" error
    # This finds columns that exist in BOTH source and target
    source_cols = set(spark.read.table(source_table).columns)
    target_cols = set(spark.read.table(target_table).columns)
    
    # Common columns (excluding metadata for the join)
    common_cols = list(source_cols.intersection(target_cols))
    join_cols = [c for c in common_cols if c not in ["load_dt", "source", "_rescued_data"]]
    
    # Build dynamic SQL fragments
    join_condition = " AND ".join([f"s.{c} <=> t.{c}" for c in join_cols])
    insert_cols = ", ".join(common_cols)
    insert_values = ", ".join([f"s.{c}" for c in common_cols])

    # 3. Execute the Idempotent Merge
    spark.sql(f"""
        MERGE INTO {target_table} t
        USING {source_table} s
        ON {join_condition}
        WHEN MATCHED THEN
          UPDATE SET t.load_dt = s.load_dt, t.source = s.source
        WHEN NOT MATCHED THEN
          INSERT ({insert_cols}) VALUES ({insert_values})
    """)
